# Polynomial Regression: SVD Optimizer Analysis

Analysis of SVD optimizer vs standard optimizers on polynomial regression task.

## 1. Setup & Data Loading

In [1]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from style import set_style, lr_labels
set_style()

PLOT_DIR = Path('plots/polynomial')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

Plots will be saved to: /Users/sambt/iaifi/sv3/analysis/plots/polynomial


In [2]:
# Load JSONL results
from style import load_results, load_results_jsonl

df = load_results_jsonl("polynomial_scan", slim=False)
print(f"Total: {len(df)} experiment runs")
print(f"Optimizers: {sorted(df['optimizer'].unique())}")
print(f"Batch sizes: {sorted(df['batch_size'].unique())}")

KeyboardInterrupt: 

In [ ]:
# Remove incomplete runs from LBFGS
def is_bad(row):
    if row['optimizer'] == 'LBFGS' and np.isnan(row['losses']['val'][-1]):
        return True
    return False
df['is_bad'] = df.apply(is_bad, axis=1)
df = df[~df['is_bad']]
print(f"After removing bad runs: {len(df)} experiment runs")

In [ ]:
# Helper functions
def get_final_loss(row, loss_type='val'):
    losses = row['losses'][loss_type]
    # Return last non-NaN value (handles LBFGS instability)
    for val in reversed(losses):
        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            return val
    return np.nan

def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Add derived columns
df['final_val_loss'] = df.apply(lambda r: get_final_loss(r, 'val'), axis=1)
df['final_train_loss'] = df.apply(lambda r: get_final_loss(r, 'train'), axis=1)
df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
df['avg_epoch_time'] = df['losses'].apply(lambda l: l.get('avg_epoch_time', np.nan))
df['avg_batch_time_train'] = df['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))

df_svd = df[(df['optimizer'] == 'SVD') & (df['lr'].str.lower() != 'polyak')].copy()
df_svd_polyak = df[(df['optimizer'] == 'SVD') & (df['lr'].str.lower() == 'polyak')].copy()
df_baseline = df[df['optimizer'] != 'SVD'].copy()

bs = sorted(df['batch_size'].unique())[0]  # 32
baseline_optimizers = sorted(df_baseline['optimizer'].unique().tolist())
k_fractions = sorted(df_svd['k_fraction'].dropna().unique())
svd_lrs = sorted(df_svd['lr'].unique())
svd_rtols = sorted(df_svd['rtol'].dropna().unique())

print(f"\nSVD: {len(df_svd)} runs")
print(f"  k_fractions: {k_fractions}")
print(f"  lrs: {svd_lrs}")
print(f"  rtols: {svd_rtols}")
print(f"\nBaseline: {len(df_baseline)} runs")
print(f"  Optimizers: {baseline_optimizers}")
print(f"  lrs: {sorted([L for L in df_baseline['lr'].unique() if L is not None])}")

## 2. Best Performance Comparison

In [ ]:
# Find best run for each optimizer
best_runs = []

best_svd = df_svd.loc[df_svd['final_val_loss'].idxmin()]
best_runs.append({
    'optimizer': 'SVD',
    'final_val_loss': best_svd['final_val_loss'],
    'final_train_loss': best_svd['final_train_loss'],
    'lr': best_svd['lr'],
    'k': best_svd['k'],
    'k_fraction': best_svd['k_fraction'],
    'rtol': best_svd['rtol'],
    'total_time': best_svd['total_time']
})

for opt in baseline_optimizers:
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best = opt_df.loc[opt_df['final_val_loss'].idxmin()]
    best_runs.append({
        'optimizer': opt,
        'final_val_loss': best['final_val_loss'],
        'final_train_loss': best['final_train_loss'],
        'lr': best['lr'],
        'k': None,
        'k_fraction': None,
        'rtol': None,
        'total_time': best['total_time']
    })

best_df = pd.DataFrame(best_runs)
print("Best performance for each optimizer:")
print(best_df.to_string(index=False))

# Custom plots

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
n_epochs = len(train_curve)
epochs_train = np.arange(1,n_epochs + 1)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):#,'AdamW','Lion','Muon','ScheduleFreeSGD','ScheduleFreeAdamW']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,5])
plt.xlim(0,n_epochs+1)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
n_epochs = len(train_curve)
epochs_train = np.arange(n_epochs)

ax.plot(epochs_train, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(epochs_train, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim(0,n_epochs)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
#plt.savefig(PLOT_DIR / 'val_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.cumsum(t)
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'train'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time (s)')
ax.set_ylabel('Train Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
#plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_train_loss_best.pdf')
plt.show()

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

# Best SVD
best_svd_row = df.loc[df_svd[df_svd['batch_size'] == bs]['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'val')
wall_times = get_loss_curve(best_svd_row, 'epoch_times')
wall_times = np.concatenate([np.array([1]), np.array(wall_times)])
t = np.cumsum(wall_times)

ax.plot(t, train_curve, color='k', linewidth=3, label=f"Sven ($\eta={best_svd_row['lr']}$, $k={int(best_svd_row['k'])}$, rtol=${lr_labels[best_svd_row['rtol']]}$)",zorder=2)

# Best baselines
for i, opt in enumerate(['SGD','PolyakSGD','RMSprop','Adam','LBFGS']):
    opt_df = df_baseline[(df_baseline['optimizer'] == opt) & (df_baseline['batch_size'] == bs)]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    t = get_loss_curve(best_row, 'epoch_times')
    t = np.concatenate([np.array([1]), np.array(t)])
    t = np.cumsum(t)
    #label = f"{opt} ($\eta={lr_labels[best_row['lr']]}$)" if opt != "PolyakSGD" else f"{opt}"
    label = f"{opt}"
    ax.plot(t, get_loss_curve(best_row, 'val'), color=colors[i], linewidth=2, label=label,zorder=1)

ax.set_xlabel('Wall Time + 1 (sec)')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1)
ax.set_title(f"Random Polynomial ($B = {bs}$)")
plt.ylim([None,2])
plt.xlim([1,None])
plt.xscale('log')
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'wall_time_vs_val_loss_best.pdf')
plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'train')
            epochs = np.arange(1, len(train_curve) + 1)
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'train'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Train Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, 1D Random Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            train_curve = get_loss_curve(row, 'val')
            epochs = np.arange(len(train_curve))
            ax.plot(epochs, train_curve, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        df_sgd = df_baseline[(df_baseline['batch_size'] == bs) & (df_baseline['optimizer'] == 'SGD')]
        best_sgd_row = df.loc[df_sgd['final_val_loss'].idxmin()]
        ax.plot(epochs, get_loss_curve(best_sgd_row, 'val'), label=f"SGD ($\eta={lr_labels[best_sgd_row['lr']]}$)", color='k', linestyle='--', linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation Loss')
        ax.set_yscale('log')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.ylim([None,3])
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"val_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
LR = 0.5
K = 64
colors = sns.color_palette("deep")
fig, ax = plt.subplots(figsize=(8, 6))
for j,K in enumerate([8,16,32]):
    df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['k'] == K)]
    rtol_values = sorted(df_sel['rtol'].unique())
    rtol_styles = {1e-4: 'dashed', 1e-3: 'solid', 1e-2: 'dashdot', 1e-1: 'dotted'}
    for i,rtol in enumerate(rtol_values):
        row = df_sel[df_sel['rtol'] == rtol]
        assert len(row) == 1, f"Expected exactly one run for rtol={rtol}, found {len(row)}"
        row = row.iloc[0]
        val_loss = row['losses']['val']
        epochs = np.arange(0, len(val_loss))
        ax.plot(epochs, val_loss, color=colors[j], linewidth=3, linestyle=rtol_styles[rtol])

rtol_lines = [Line2D([0], [0], color='k', linestyle=rtol_styles[rtol], linewidth=3) for rtol in rtol_values]
rtol_labels_legend = [f"${lr_labels[rtol]}$" for rtol in rtol_values]
custom_lines = [Line2D([0], [0], color=colors[j], linewidth=3) for j in range(3)]
custom_labels = [f"$k={k}$" for k in [16,32,64]]
leg1 = plt.legend(custom_lines, custom_labels, loc='upper right', ncol=1, frameon=True, framealpha=1, bbox_to_anchor=(0.84, 0.945))
plt.gca().add_artist(leg1)
leg2 = plt.legend(rtol_lines, rtol_labels_legend, loc='upper right', frameon=True, framealpha=1, title='rtol')
plt.gca().add_artist(leg2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Number of SVs Used by Sven')
ax.set_title(f"Sven, Polynomial ($B = {bs}$, $\eta={LR}$)")
plt.xlim(0,n_epochs+1)
plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
plt.grid(axis='both', linestyle='--', alpha=0.7)
plt.yscale('log')
plt.ylim([None,0.5])

plt.tight_layout()
#plt.savefig(PLOT_DIR / f"val_loss_overlay_rtol_and_k.pdf")
plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
    for RTOL in [1e-4,1e-3,1e-2]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR) & (df_svd['rtol'] == RTOL)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate(k_values):
            row = df_sel[df_sel['k'] == k]
            assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
            row = row.iloc[0]
            nonzero_k = np.array(row['svd_info']['num_nonzero_svs'])
            n_epoch = len(get_loss_curve(row, 'train'))
            nonzero_k = nonzero_k.reshape(n_epoch,-1).mean(axis=1)
            epochs = np.arange(1, n_epoch + 1)
            ax.plot(epochs, nonzero_k, label=f"$k={int(k)}$", color=colors[i], linewidth=3)

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Number of SVs Used by Sven')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, MNIST ($B = {bs}$, $\eta={LR}$, rtol = ${lr_labels[RTOL]})$")
        plt.xlim(0,n_epochs+1)
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        #plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

In [ ]:
for LR in [0.05,0.1,0.5]:
        fig, ax = plt.subplots(figsize=(8, 6))
        df_sel = df_svd[(df_svd['batch_size'] == bs) & (df_svd['lr'] == LR)]
        k_values = sorted(df_sel['k'].unique())
        colors = sns.color_palette("viridis",n_colors=len(k_values))
        for i,k in enumerate([32]):
            styles = ['-', '--', ':']
            for j,RTOL in enumerate([1e-4,1e-3,1e-2]):
                row = df_sel[(df_sel['rtol'] == RTOL) & (df_sel['k'] == k)]
                assert len(row) == 1, f"Expected exactly one run for k={k}, found {len(row)}"
                row = row.iloc[0]
                nonzero_k = np.array(row['svd_info']['num_nonzero_svs'])
                n_epoch = len(get_loss_curve(row, 'train'))
                nonzero_k = nonzero_k.reshape(n_epoch,-1).mean(axis=1)
                epochs = np.arange(1, n_epoch + 1)
                ax.plot(epochs, nonzero_k, label=f"$k={int(k)}$, rtol = ${lr_labels[RTOL]}$", color=colors[i], linewidth=3, linestyle=styles[j])

        ax.set_xlabel('Epoch')
        ax.set_ylabel('Number of SVs Used by Sven')
        plt.legend(ncol=3,loc='upper center',frameon=True,framealpha=1)
        ax.set_title(f"Sven, MNIST ($B = {bs}$, $\eta={LR}$)")
        plt.xlim(0,n_epochs+1)
        plt.ylim([0,35])
        plt.xticks(np.arange(0, n_epochs+1, max(1, n_epochs//10)))
        plt.grid(axis='both', linestyle='--', alpha=0.7)

        plt.tight_layout()
        #plt.savefig(PLOT_DIR / f"train_loss_k_comparison_bs{bs}_lr{LR}_rtol{RTOL}.pdf")
        plt.show()

# Bulk plots

In [ ]:
# Bar chart: best final validation loss by optimizer
fig, ax = plt.subplots(figsize=(7, 5))

colors = ['C0', 'C1', 'C2', 'C3']
x = np.arange(len(best_df))
bars = ax.bar(x, best_df['final_val_loss'], color=colors)

ax.set_xticks(x)
ax.set_xticklabels(best_df['optimizer'])
ax.set_ylabel('Final Validation Loss')
ax.set_title('Best Final Validation Loss by Optimizer')
ax.set_yscale('log')

for bar, val in zip(bars, best_df['final_val_loss']):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.2e}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'best_optimizer_comparison.pdf')
plt.show()

## 3. Training & Validation Curves: Best Configurations

In [ ]:
# Train + val loss curves for best config of each optimizer
fig, ax = plt.subplots(figsize=(8, 6))
legend_labels = []

best_svd_row = df.loc[df_svd['final_val_loss'].idxmin()]
train_curve = get_loss_curve(best_svd_row, 'train')
val_curve = get_loss_curve(best_svd_row, 'val')
n_epochs = len(train_curve)
epochs_train = np.arange(1, n_epochs + 1)
epochs_val = np.arange(len(val_curve))

ax.plot(epochs_train, train_curve, 'C0-')
ax.plot(epochs_val, val_curve, 'C0--')
legend_labels.append(f"SVD (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])})")

for i, opt in enumerate(baseline_optimizers):
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    ax.plot(epochs_train, get_loss_curve(best_row, 'train'), f'C{i+1}-')
    ax.plot(epochs_val, get_loss_curve(best_row, 'val'), f'C{i+1}--')
    legend_labels.append(f"{opt} (lr={best_row['lr']:.0e})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_yscale('log')
ax.set_title('Polynomial Regression (bs=32)')

leg1 = ax.legend(handles=[Line2D([], [], color=f'C{i}', linestyle='-', label=legend_labels[i])
                          for i in range(len(legend_labels))], loc='upper right')
ax.add_artist(leg1)
ax.legend(handles=[Line2D([], [], color='k', linestyle='-', label='Train'),
                   Line2D([], [], color='k', linestyle='--', label='Val')], loc='lower left')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'train_val_loss_best.pdf')
plt.show()

In [ ]:
# Batchwise training loss for best configs
fig, ax = plt.subplots(figsize=(8, 6))

best_svd_row = df.loc[df_svd['final_val_loss'].idxmin()]
bl = get_loss_curve(best_svd_row, 'train_batch')
ex = np.linspace(0, n_epochs, len(bl))
ax.plot(ex, bl, 'C0-', alpha=0.7,
        label=f"SVD (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])})")

for i, opt in enumerate(baseline_optimizers):
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    bl = get_loss_curve(best_row, 'train_batch')
    ex = np.linspace(0, n_epochs, len(bl))
    ax.plot(ex, bl, f'C{i+1}-', alpha=0.7,
            label=f"{opt} (lr={best_row['lr']:.0e})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_yscale('log')
ax.set_title('Polynomial: Batch-level Training Loss')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'training_loss_batchwise_best.pdf')
plt.show()

## 4. SVD Hyperparameter Sensitivity

In [ ]:
# Heatmap: final val loss vs k_fraction and lr, per rtol
fig, axes = plt.subplots(1, len(svd_rtols), figsize=(4*len(svd_rtols), 4))
if len(svd_rtols) == 1:
    axes = [axes]

for ax, rtol in zip(axes, svd_rtols):
    data = df_svd[df_svd['rtol'] == rtol]
    pivot = data.pivot_table(values='final_val_loss', index='k_fraction', columns='lr', aggfunc='first')
    im = ax.imshow(np.log10(pivot.values), aspect='auto', cmap='viridis',
                   vmin=np.log10(df_svd['final_val_loss'].min()),
                   vmax=np.log10(df_svd['final_val_loss'].max()))
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{lr}' for lr in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{int(kf*bs)}' for kf in pivot.index])
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('k')
    ax.set_title(f'rtol = {rtol}')
    if ax != axes[0]:
        ax.set_ylabel('')
        ax.set_yticklabels([])

fig.subplots_adjust(right=0.85)
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('log10(Validation Loss)')

plt.savefig(PLOT_DIR / 'svd_heatmap_by_rtol.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Learning rate sensitivity at each k (best across rtol)
fig, ax = plt.subplots(figsize=(8, 6))
for i, kf in enumerate(k_fractions):
    kf_data = df_svd[df_svd['k_fraction'] == kf]
    best_per_lr = kf_data.groupby('lr')['final_val_loss'].min().reset_index()
    ax.plot(best_per_lr['lr'], best_per_lr['final_val_loss'], 'o-',
            label=f'k={int(kf*bs)}', color=f'C{i}')

ax.set_xlabel('Learning Rate')
ax.set_ylabel('Final Validation Loss')
ax.set_title(f'LR Sensitivity (bs={bs})')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'lr_sensitivity.pdf')
plt.show()

In [ ]:
# k effect at each lr
for lr in svd_lrs:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, rtol in enumerate(svd_rtols):
        data = df_svd[(df_svd['lr'] == lr) & (df_svd['rtol'] == rtol)]
        data = data.sort_values('k_fraction')
        ax.plot(data['k_fraction'] * bs, data['final_val_loss'], 'o-',
                label=f'rtol={rtol}', color=f'C{i}')
    ax.set_xlabel('k')
    ax.set_ylabel('Final Validation Loss')
    ax.set_title(f'k Effect (lr={lr})')
    ax.set_yscale('log')
    ax.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'k_fraction_effect_lr{lr}.pdf')
    plt.show()

In [ ]:
# rtol effect at each lr
for lr in svd_lrs:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, kf in enumerate(k_fractions):
        data = df_svd[(df_svd['lr'] == lr) & (df_svd['k_fraction'] == kf)]
        data = data.sort_values('rtol')
        ax.plot(data['rtol'], data['final_val_loss'], 'o-',
                label=f'k={int(kf*bs)}', color=f'C{i}')
    ax.set_xlabel('rtol')
    ax.set_ylabel('Final Validation Loss')
    ax.set_title(f'rtol Effect (lr={lr})')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'rtol_effect_lr{lr}.pdf')
    plt.show()

## 5. SVD Batchwise Training Curves

In [ ]:
# Fix lr and rtol, compare k values
for lr in svd_lrs:
    for rtol in svd_rtols:
        fig, ax = plt.subplots(figsize=(8, 6))
        data = df_svd[(df_svd['lr'] == lr) & (df_svd['rtol'] == rtol)]
        #for i, kf in enumerate(k_fractions):
        #    row = data[data['k_fraction'] == kf]
        #    if len(row) == 0:
        #        continue
        #    row = row.iloc[0]
        #    bl = row['losses']['train_batch']
        #    ax.plot(bl, f'C{i}-', lw=1.5, label=f'k={int(kf*bs)}')
        ax.set_xlabel('Batch')
        ax.set_ylabel('Training Loss')
        ax.set_title(f'lr={lr}, rtol={rtol}')
        ax.set_yscale('log')
        ax.legend()
        plt.tight_layout()
        smooth_batches = 50
        for i, kf in enumerate(k_fractions):
            row = data[data['k_fraction'] == kf]
            if len(row) == 0:
                continue
            row = row.iloc[0]
            bl = np.array(row['losses']['train_batch'])
            nbatch = len(bl)
            if len(bl) >= smooth_batches:
                bl = sliding_average(bl, window=smooth_batches)
            xvals = np.linspace(0, nbatch, len(bl))
            ax.plot(xvals, bl, f'C{i}-', lw=1.5, label=f'k={int(kf*bs)}')
        ax.set_title(f'lr={lr}, rtol={rtol}, {smooth_batches}-batch smoothing')
        plt.savefig(PLOT_DIR / f'svd_batchloss_kfScan_lr{lr}_rtol{rtol}.pdf')
        plt.show()

In [ ]:
# Fix k and rtol, compare lrs
for kf in k_fractions:
    for rtol in svd_rtols:
        fig, ax = plt.subplots(figsize=(8, 6))
        data = df_svd[(df_svd['k_fraction'] == kf) & (df_svd['rtol'] == rtol)]
        for i, lr in enumerate(svd_lrs):
            row = data[data['lr'] == lr]
            if len(row) == 0:
                continue
            row = row.iloc[0]
            bl = row['losses']['train_batch']
            ax.plot(bl, f'C{i}-', lw=1.5, label=f'lr={lr}')
        ax.set_xlabel('Batch')
        ax.set_ylabel('Training Loss')
        ax.set_title(f'k={int(kf*bs)}, rtol={rtol}')
        ax.set_yscale('log')
        ax.legend()
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'svd_batchloss_lrScan_kf{kf}_rtol{rtol}.pdf')
        plt.show()

## 6. Singular Value Analysis

In [ ]:
# Number of nonzero SVs over training
smooth = 20

for lr in svd_lrs:
    for rtol in svd_rtols:
        fig, ax = plt.subplots(figsize=(8, 6))
        data = df_svd[(df_svd['lr'] == lr) & (df_svd['rtol'] == rtol)]
        legend_entries = []
        for i, kf in enumerate(k_fractions):
            row = data[data['k_fraction'] == kf]
            if len(row) == 0:
                continue
            row = row.iloc[0]
            if row['svd_info'] is None:
                continue
            num_nonzero = row['svd_info']['num_nonzero_svs']
            n_ep = len(row['losses']['train'])
            x = np.linspace(0, n_ep, len(num_nonzero))
            ax.plot(x, num_nonzero, f'C{i}-', lw=1, alpha=0.25)
            n_smooth = sliding_average(num_nonzero, window=smooth)
            ax.plot(x[smooth-1:], n_smooth, f'C{i}-', lw=2)
            legend_entries.append(Line2D([], [], label=f'k={int(kf*bs)}', color=f'C{i}'))

        ax.set_xlabel('Epoch')
        ax.set_ylabel('# Nonzero Singular Values')
        ax.set_title(f'lr={lr}, rtol={rtol}')
        if legend_entries:
            ax.legend(handles=legend_entries)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'num_nonzero_svs_lr{lr}_rtol{rtol}.pdf')
        plt.show()

In [ ]:
# Dual axis: nonzero SVs + training loss for best SVD config
best_svd_row = df.loc[df_svd['final_val_loss'].idxmin()]

fig, ax = plt.subplots(figsize=(8, 6))
ax2 = ax.twinx()

num_nonzero = best_svd_row['svd_info']['num_nonzero_svs']
train_batch = best_svd_row['losses']['train_batch']
n_ep = len(best_svd_row['losses']['train'])
x = np.linspace(0, n_ep, len(num_nonzero))

smooth_frac = 0.1
sw = max(1, int(smooth_frac * len(num_nonzero) / n_ep))
nn_smooth = sliding_average(num_nonzero, window=sw)
tl_smooth = sliding_average(train_batch, window=sw)

ax.plot(x[sw-1:], nn_smooth, 'C0-', lw=2)
ax2.plot(x[sw-1:], tl_smooth, 'C1--', lw=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('# Nonzero Singular Values', color='C0')
ax2.set_ylabel('Training Loss', color='C1')
ax2.set_yscale('log')
ax.set_title(f"Best SVD: lr={best_svd_row['lr']}, k={int(best_svd_row['k'])}, rtol={best_svd_row['rtol']}")

ax.legend(handles=[Line2D([], [], color='C0', linestyle='-', label='# Nonzero SVs'),
                   Line2D([], [], color='C1', linestyle='--', label='Training Loss')],
          loc='upper right')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'nonzero_svs_vs_trainloss_best.pdf')
plt.show()

In [ ]:
# SV spectrum evolution for best config
best_svd_row = df.loc[df_svd['final_val_loss'].idxmin()]
svs_list = best_svd_row['svd_info']['svs']
n_ep = len(best_svd_row['losses']['train'])

if svs_list is not None and len(svs_list) > 0:
    total_batches = len(svs_list)
    bpe = total_batches // n_ep
    sample_points = [
        (0, 'Batch 0 (Start)'),
        (bpe, 'Epoch 1'),
        (3*bpe, 'Epoch 3'),
        (n_ep//2*bpe, f'Epoch {n_ep//2}'),
        (3*n_ep//4*bpe, f'Epoch {3*n_ep//4}'),
        (total_batches - 1, f'Epoch {n_ep} (End)'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, (bidx, title) in zip(axes.flat, sample_points):
        svs = svs_list[min(bidx, len(svs_list)-1)]
        ax.bar(range(len(svs)), sorted(svs, reverse=True), color='C0', alpha=0.7)
        ax.set_xlabel('SV Index')
        ax.set_ylabel('Singular Value')
        ax.set_title(title)
        ax.set_yscale('log')

    plt.suptitle(f"SV Spectrum Evolution (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])}, rtol={best_svd_row['rtol']})", fontsize=14)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'sv_spectrum_evolution.pdf')
    plt.show()

## 7. Wall-Time Analysis

In [ ]:
# Total wall time comparison for best configs
fig, ax = plt.subplots(figsize=(7, 5))

colors = ['C0', 'C1', 'C2', 'C3']
x = np.arange(len(best_df))
bars = ax.bar(x, best_df['total_time'], color=colors)

ax.set_xticks(x)
ax.set_xticklabels(best_df['optimizer'])
ax.set_ylabel('Total Time (s)')
ax.set_title('Total Training Time (Best Configs)')

for bar, val in zip(bars, best_df['total_time']):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.1f}s',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'total_time_best_configs.pdf')
plt.show()

In [ ]:
# Average batch time comparison
fig, ax = plt.subplots(figsize=(7, 5))

batch_times = []
labels = []

best_svd_row = df.loc[df_svd['final_val_loss'].idxmin()]
batch_times.append(best_svd_row['avg_batch_time_train'])
labels.append('SVD')

for opt in baseline_optimizers:
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    batch_times.append(best_row['avg_batch_time_train'])
    labels.append(opt)

x = np.arange(len(labels))
bars = ax.bar(x, [t*1000 for t in batch_times], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Avg Batch Time (ms)')
ax.set_title('Average Training Batch Time (Best Configs)')

for bar, val in zip(bars, batch_times):
    ax.text(bar.get_x() + bar.get_width()/2, val*1000, f'{val*1000:.2f}ms',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'avg_batch_time_best_configs.pdf')
plt.show()

In [ ]:
# Per-epoch time over training for best configs
fig, ax = plt.subplots(figsize=(8, 6))

best_svd_row = df.loc[df_svd['final_val_loss'].idxmin()]
epoch_times = best_svd_row['losses']['epoch_times']
ax.plot(range(1, len(epoch_times)+1), epoch_times, 'C0o-',
        label=f"SVD (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])})")

for i, opt in enumerate(baseline_optimizers):
    opt_df = df_baseline[df_baseline['optimizer'] == opt]
    best_row = df.loc[opt_df['final_val_loss'].idxmin()]
    et = best_row['losses']['epoch_times']
    ax.plot(range(1, len(et)+1), et, f'C{i+1}o-',
            label=f"{opt} (lr={best_row['lr']:.0e})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Epoch Time (s)')
ax.set_title('Per-Epoch Wall Time (Best Configs)')
ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'epoch_time_best_configs.pdf')
plt.show()

In [ ]:
# Loss vs wall time for best configs
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, loss_type, title in zip(axes, ['train', 'val'], ['Training Loss', 'Validation Loss']):
    best_svd_row = df.loc[df_svd['final_val_loss'].idxmin()]
    epoch_times = best_svd_row['losses']['epoch_times']
    cum_time = np.cumsum(epoch_times)
    losses = get_loss_curve(best_svd_row, loss_type)
    if loss_type == 'val':
        cum_time_plot = np.concatenate([[0], cum_time])
    else:
        cum_time_plot = cum_time
    ax.plot(cum_time_plot, losses, 'C0-',
            label=f"SVD (lr={best_svd_row['lr']}, k={int(best_svd_row['k'])})")

    for i, opt in enumerate(baseline_optimizers):
        opt_df = df_baseline[df_baseline['optimizer'] == opt]
        best_row = df.loc[opt_df['final_val_loss'].idxmin()]
        et = best_row['losses']['epoch_times']
        ct = np.cumsum(et)
        l = get_loss_curve(best_row, loss_type)
        if loss_type == 'val':
            ct_plot = np.concatenate([[0], ct])
        else:
            ct_plot = ct
        ax.plot(ct_plot, l, f'C{i+1}-',
                label=f"{opt} (lr={best_row['lr']:.0e})")

    ax.set_xlabel('Wall Time (s)')
    ax.set_ylabel(title)
    ax.set_yscale('log')
    ax.set_title(f'{title} vs Wall Time')
    ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'loss_vs_walltime_best.pdf')
plt.show()

In [ ]:
# SVD wall time vs k
fig, ax = plt.subplots(figsize=(8, 6))

for i, lr in enumerate(svd_lrs):
    data = df_svd[df_svd['lr'] == lr]
    avg_per_kf = data.groupby('k_fraction')['total_time'].mean().reset_index()
    ax.plot(avg_per_kf['k_fraction'] * bs, avg_per_kf['total_time'], 'o-',
            label=f'lr={lr}', color=f'C{i}')

ax.set_xlabel('k')
ax.set_ylabel('Total Training Time (s)')
ax.set_title('SVD Wall Time vs k')
ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'svd_walltime_vs_kfraction.pdf')
plt.show()

In [ ]:
# Efficiency: final val loss vs total time scatter
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(df_svd['total_time'], df_svd['final_val_loss'],
           c='C0', alpha=0.6, label='SVD', s=50)

for i, opt in enumerate(baseline_optimizers):
    opt_data = df_baseline[df_baseline['optimizer'] == opt]
    ax.scatter(opt_data['total_time'], opt_data['final_val_loss'],
               c=f'C{i+1}', alpha=0.6, label=opt, s=50, marker='s')

ax.set_xlabel('Total Training Time (s)')
ax.set_ylabel('Final Validation Loss')
ax.set_yscale('log')
ax.set_title('Efficiency: Loss vs Wall Time (all runs)')
ax.legend()

plt.tight_layout()
plt.savefig(PLOT_DIR / 'efficiency_loss_vs_walltime.pdf')
plt.show()

## 8. Summary

In [ ]:
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

print("\nBest performance by optimizer:")
print(best_df.to_string(index=False))

print("\n" + "-" * 60)
print(f"\nSVD optimizer ({len(df_svd)} runs):")
print(f"  Final val loss:  min={df_svd['final_val_loss'].min():.2e}, "
      f"median={df_svd['final_val_loss'].median():.2e}, "
      f"max={df_svd['final_val_loss'].max():.2e}")
print(f"  Total time:      min={df_svd['total_time'].min():.1f}s, "
      f"median={df_svd['total_time'].median():.1f}s, "
      f"max={df_svd['total_time'].max():.1f}s")

print(f"\nBaseline optimizers:")
for opt in baseline_optimizers:
    od = df_baseline[df_baseline['optimizer'] == opt]
    print(f"  {opt} ({len(od)} runs): "
          f"val_loss min={od['final_val_loss'].min():.2e}, "
          f"time min={od['total_time'].min():.1f}s")

print(f"\nAll plots saved to: {PLOT_DIR.resolve()}")